# ÍSKOS 2024 — 01. Skoðum gögnin

Áður en við byggjum líkan kynnumst við gögnunum — **ein breyta í einu**, og svo tengsl við flokksval. Mynstrið er einfalt: fyrst eitt **vinnandi dæmi** í kóða (svo þú sjáir nákvæmlega hvað gerist), svo sama rökfræði í **lykkju** yfir allar breytur.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import iskos_style as ik   # ÍSKOS litir + þema (iskos_style.py) — sami stíll og rsetup.R

PROJECT = Path.cwd().parent
DATA = PROJECT / "data"

df = pd.read_csv(DATA / "iskos_clean.csv")
# Index meta by the Icelandic question text — that's also the column name in `df`.
meta = pd.read_csv(DATA / "feature_meta.csv").set_index("label")
sns.set_theme(style="whitegrid", font_scale=0.9)
ik.apply_rc()   # ÍSKOS bakgrunnur + letur fyrir allar myndir (kallað eftir set_theme)
print(f"{len(df)} svarendur · {df.shape[1] - 1} breytur (+ flokkur)")

## 1. Flokksskipting í úrtakinu

Hvernig dreifast 7 flokksvalin meðal þeirra ÍsKos-svarenda sem við héldum eftir í forvinnslunni?

In [ ]:
counts = df["flokkur"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4.5))
# Litur hverrar súlu = flokkslitur (sjá class_colors í iskos_style.py).
ax.barh(counts.index, counts.values, color=ik.class_colors(counts.index))
ax.invert_yaxis()
ax.set_xlabel("Fjöldi svarenda")
for y, v in enumerate(counts.values):
    ax.text(v + 2, y, str(v), va="center", fontsize=9, color=ik.ISKOS_TEXT_MID)
ik.style_iskos(ax, title=f"Flokksskipting í ÍsKos bálknum (n={len(df)})")
ax.set_xticks([])
plt.tight_layout()
plt.show()

## 2. Dreifing einnar breytu — vinnandi dæmi: `gender`

Áður en við gerum þetta á allar breytur skoðum við eina vandlega. Fyrir hverja breytu viljum við sjá:

- Heitið og íslenska lýsingu úr `feature_meta`.
- Hópinn sem hún tilheyrir.
- Dreifingu gildanna: súlurit fyrir flokkabreytur, stuðlarit fyrir raðbreytur.
- Hversu margir svöruðu ekki (`„Ekki svarað“` eða `-1`).

In [ ]:
def plot_distribution(question: str):
    info = meta.loc[question]
    dtype = info["dtype"]

    fig, ax = plt.subplots(figsize=(8, 3.4))
    bar_color = ik.ISKOS_MAIN          # einn meginlitur fyrir staka dreifingu
    text_color = ik.ISKOS_TEXT_MID

    if dtype == "nominal":
        vc = df[question].value_counts()
        values = vc.values
        bars = ax.barh(vc.index.astype(str), values, color=bar_color)
        ax.invert_yaxis()

        max_val = values.max()
        for bar, val in zip(bars, values):
            ax.text(bar.get_width() + max_val * 0.01,
                    bar.get_y() + bar.get_height() / 2,
                    f"{val:,}", va="center", ha="left",
                    fontsize=9, color=text_color)

        ax.set_xlim(0, max_val * 1.12)
        ax.set_xticks([])
        ax.tick_params(axis="y", length=0, labelsize=9, colors=text_color)

    else:
        s = df[question].dropna().astype(int)
        full_range = range(int(s.min()), int(s.max()) + 1)
        vc = s.value_counts().reindex(full_range, fill_value=0).sort_index()
        positions = list(vc.index)
        values = vc.values

        bars = ax.bar(positions, values, color=bar_color, width=0.8)
        ax.set_xticks(positions)
        ax.set_xticklabels([str(p) for p in positions])

        max_val = values.max()
        for bar, val in zip(bars, values):
            if val == 0:
                continue
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max_val * 0.02,
                    f"{val:,}", ha="center", va="bottom",
                    fontsize=9, color=text_color)

        ax.set_ylim(0, max_val * 1.15)
        ax.set_yticks([])
        ax.tick_params(axis="x", length=0, labelsize=9, colors=text_color)

    ik.style_iskos(ax, title=question)

    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.tight_layout()
    plt.show()

# Find gender's question text — we don't hard-code Icelandic strings, we look them up.
gender_q = meta[meta["short_name"] == "gender"].index[0]
plot_distribution(gender_q)

## 3. Sama rökfræði á allar breytur

Nú lykkjum við yfir öll 61 atriðin í `feature_meta` og notum sömu fall. Þetta er löng sería — skrunaðu í gegn til að átta þig á þeim variables sem skipta máli.

In [ ]:
for question in meta.index:
    plot_distribution(question)

## 4. Tengsl við flokksval — vinnandi dæmi (flokkabreyta): `gender`

Fyrir hverja flokkabreytu sýnum við **stafla súlna**: hver flokkur svarenda (t.d. „Kona“, „Karl“) fær eina súlu, og súlan er skipt í liti eftir flokksvali. Hæð á einum lit innan súlu sýnir hlutfall fólks í þeim hópi sem kaus þann flokk.

Ef litir liggja jafnt yfir allar súlur ⇒ flokkabreytan greinir ekki á milli flokksvala. Ef litirnir skiptast skarpt eftir hópi ⇒ sterkur forspáríkur.

In [ ]:
# Stable y-axis ordering so the same party row is in the same place across all plots.
FLOKKUR_ORDER = sorted(df["flokkur"].unique().tolist())

def plot_vs_target(question: str):
    info = meta.loc[question]
    dtype = info["dtype"]

    fig, ax = plt.subplots(figsize=(9, 4))
    text_color = ik.ISKOS_TEXT_MID
    bar_color = ik.ISKOS_MAIN

    if dtype == "nominal":
        ct = pd.crosstab(df[question], df["flokkur"], normalize="index")[FLOKKUR_ORDER]
        # Staflarnir eru litaðir eftir flokki (party_colors via class_colors).
        ct.plot(kind="bar", stacked=True, ax=ax,
                color=ik.class_colors(FLOKKUR_ORDER), width=0.85)
        ax.set_ylabel("Hlutfall innan hóps", fontsize=9, color=text_color)
        ax.set_xlabel("")
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left",
                  fontsize=7, title="flokkur", frameon=False)
        plt.setp(ax.get_xticklabels(), rotation=30, ha="right",
                 fontsize=9, color=text_color)
    else:
        # Boxplot per class — shows median + IQR + whiskers without the 989-dot pile-up
        # the strip plot produces on discrete 0..10 / 1..5 Likert scales.
        d = df[df[question] != -1]
        sns.boxplot(
            data=d, x=question, y="flokkur", order=FLOKKUR_ORDER,
            color=bar_color, ax=ax, width=0.55, fliersize=2.5, linewidth=1.0,
            boxprops={"alpha": 0.55, "edgecolor": text_color},
            medianprops={"color": "#222222", "linewidth": 1.6},
            whiskerprops={"color": text_color},
            capprops={"color": text_color},
        )
        if len(d):
            means = d.groupby("flokkur")[question].mean()
            xs = [means.get(lbl, float("nan")) for lbl in FLOKKUR_ORDER]
            ax.scatter(xs, range(len(FLOKKUR_ORDER)),
                       color="black", s=35, marker="D", zorder=10, label="meðaltal")
            ax.legend(loc="upper right", fontsize=8, frameon=False)
        ax.set_xlabel("")
        ax.set_ylabel("")

    ik.style_iskos(ax, title=question)
    ax.tick_params(axis="both", length=0, labelsize=9, colors=text_color)
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.tight_layout()
    plt.show()

plot_vs_target(gender_q)

## 5. `lrscale` sem raðbreytudæmi

Fyrir raðbreytur sýnum við **kassagraf** (boxplot) eitt fyrir hvern flokk: kassinn nær frá 1. til 3. fjórðungsmarka (IQR), lóðrétt strik er miðgildi, og **svartur tígull** merkir meðaltalið. Dílarnir til hliðar sýna útbreiðsluna á stökum sem teljast útlagar (e. outliers). Ef kassarnir og tíglarnir liggja á mismunandi stöðum á x-ásnum þýðir það að breytan skilur flokka að.

In [ ]:
lrscale_q = meta[meta["short_name"] == "lrscale"].index[0]
plot_vs_target(lrscale_q)

## 6. Sama rökfræði á allar breytur

Lykkja yfir alla 61 eiginleika. Breytur þar sem litir skiptast skarpt (flokkabreytur) eða þar sem flokksmeðaltöl dreifast vel um x-ásinn (raðbreytur) eru þær sem líkanið mun nýta mest. Hafðu þetta í huga þegar þú lest niðurstöður næstu vinnubókar.

In [ ]:
for question in meta.index:
    plot_vs_target(question)